# EXP-031 — DenseNet121 + lesion-evidence gate (late fusion)

**Trains nothing.** It reuses the frozen EXP-029 softmax outputs and adds one
post-hoc decision layer driven by YOLO lesion evidence at the **image** level.

## The rule

> If the classifier's argmax is a gated disease class but the detector finds no
> lesion of that class at confidence `>= TAU`, fall back to the classifier's
> next-best class.

Many boxes per image is exactly what this design wants: the boxes are reduced
to per-image statistics (`max_conf`, `n_boxes`, `area_frac` per class), so
there is no cropping, no per-crop classification and no aggregation ambiguity.

## Why not crop -> classify

The annotation guideline bounds **lesions**, not leaves, and `healthy` /
`not_cashew_leaf` carry **zero boxes** by policy. A healthy image therefore
produces no crops at all, so a crop pipeline structurally cannot classify two
of the five classes. Lesion crops are also a completely different input
distribution from whole leaves, which would force a classifier retrain whose
output would duplicate the detector's own class head.

## Why this should work

Across the five EXP-029 seeds, **77 of 109** holdout errors are *predicted
anthracnose*; anthracnose receives **1.209x** more predictions than its true
support while healthy receives **0.758x**. An oracle gate reaches **92.38%**
on Holdout 09 against EXP-029's 82.70%. Test is 96.23%, so the 10-point gap
target needs Holdout >= **86.23%**.

The detector supports this: `CASHEW_YOLO_MASTER_FINAL_NEG_V03` carries ~53
boxes per positive image (14,507 anthracnose boxes over 893 positive
Validation images), so image-level sensitivity should sit far above the 0.49
box-level recall — the gate only needs **one** box. And YOLO26s NEG V03 raises
just 3 false alarms on 58 negatives at `conf >= 0.25`; both healthy ones are
anthracnose at confidence 0.297 and 0.311, so a TAU above ~0.35 would clear
them entirely.

## Protocol

1. Leakage audit: no Holdout 09 image may appear in the detector's training
   split. Hard abort if it does.
2. `TAU` is calibrated on Split (13) **Validation only**, after removing any
   image the detector trained on.
3. Holdout 09 is scored once, at the end.
4. The gate is applied to **both** Test and Holdout, and the reported gap is
   `gated_Test - gated_Holdout`. Gating only the holdout side would overstate
   the result.

**Before running:** EXP-029 prediction tables are embedded and checksum-verified.
Missing Split 13, Holdout 09 and YOLO NEG V03 image folders are staged from Drive.
The verified YOLO26s `best.pt` is loaded from an immutable GitHub commit
when absent locally; `DETECTOR_WEIGHTS_OVERRIDE` remains available.


In [1]:
# ============================================================
# 00. IMPORTS AND PATHS
# ============================================================
import gc, hashlib, json, os, sys, time, platform, zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy import stats
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from IPython.display import display

try:
    import ultralytics
except ImportError:
    !pip install -q ultralytics
    import ultralytics

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str),
                          encoding="utf-8")

print("pandas", pd.__version__, "| ultralytics", ultralytics.__version__)


pandas 2.2.3 | ultralytics 8.4.198


In [2]:
# ============================================================
# 01. CONFIGURATION — EXP-031 (late-fusion lesion-evidence gate)
# ============================================================
#
# EXP-031 trains NOTHING. It consumes the frozen EXP-029 classifier
# probabilities and adds one post-hoc decision layer driven by YOLO lesion
# evidence at the IMAGE level, which is why many boxes per image are fine.
#
# Rule: if the classifier's argmax is a gated disease class but YOLO finds no
# lesion of that class at confidence >= TAU, fall back to the classifier's
# next-best class. Nothing else about the classifier changes.
#
# Why this rule: across the five EXP-029 seeds, 77 of 109 holdout errors are
# "predicted anthracnose", and anthracnose receives 1.209x more predictions
# than its true support while healthy receives 0.758x. An oracle version of
# this gate reaches 92.38% on Holdout 09 (+9.68 points over EXP-029's 82.70%),
# and Test is 96.23%, so the gap target of 10 points needs Holdout >= 86.23%.
#
# Protocol guards, in order, all enforced by the cells below:
#   1. Leakage audit must PASS before any gate is calibrated.
#   2. TAU is calibrated on Split (13) Validation images only, after removing
#      any image that appears in the YOLO training split.
#   3. Holdout 09 is scored once, at the end, and never informs TAU.
#   4. The gap is reported as gated_Test - gated_Holdout. Gating is applied to
#      BOTH sides. A gate that lifts Test as much as Holdout closes nothing,
#      and reporting only the holdout side would overstate the result.

EXPERIMENT_ID = "EXP-DENSENET121-YOLOGATE-031"
NOTEBOOK_VERSION = "031"
DRIVE_TARGET_FOLDER_ID = "1DGKTnzsvVc60RIEn5lQgflR3s-loM24G"
UPLOAD_RESULT_ZIPS_TO_DRIVE = True
BASELINE_EXPERIMENT_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-029-256px-9c23ac05"

# ------------------------------------------------------------
# Inputs. Fill the Drive file IDs the same way EXP-028/029 did.
# ------------------------------------------------------------
CLASSIFIER_ANALYSIS_ZIP = "EXP-DENSENET121-TRANSFER-5SEEDS-029-256px-9c23ac05_ANALYSIS.zip"
CLASSIFIER_ANALYSIS_FILE_ID = None          # unused when embedded tables are available
CLASSIFIER_ANALYSIS_OVERRIDE = None         # optional extracted folder or ANALYSIS ZIP path

DETECTOR_NAME = "EXP-Y26S-NEG-V03-001-CLASS-WEIGHTED"
# best.pt now ships inside the FULL ZIP committed to the repo at
# training_results/negative_evaluation/EXP-Y26S-NEG-V03-001_FULL.zip
# (zip sha256 e1d3cc42afd356aaf3431486cb6f518344582e836872a36bfb651195deaf542d),
# member train/weights/best.pt. Verified locally: 20,316,608 bytes, valid
# torch archive, names = [anthracnose, leaf_miner, red_rust], model yolo26s.
DETECTOR_FULL_ZIP = "EXP-Y26S-NEG-V03-001_FULL.zip"
DETECTOR_FULL_ZIP_URL = "https://raw.githubusercontent.com/klongnguyen/KLTN_Cashew_Leaf_Disease_Classification/ac1efe16319052b6f08a3c41b4ec361bbbb19e2f/training_results/negative_evaluation/EXP-Y26S-NEG-V03-001_FULL.zip"
DETECTOR_FULL_ZIP_EXPECTED_BYTES = 51515079
DETECTOR_FULL_ZIP_SHA256 = "e1d3cc42afd356aaf3431486cb6f518344582e836872a36bfb651195deaf542d"
DETECTOR_WEIGHTS_MEMBER = "train/weights/best.pt"
DETECTOR_WEIGHTS_SHA256 = "66717b5523ba394443495378a326da26f0d24e98a331f98480f79d9d6fa18166"
DETECTOR_WEIGHTS_OVERRIDE = None            # exact local path to best.pt, if already extracted

# The detector's own dataset. Needed for the leakage audit. The TAU
# calibration does NOT need it: image-level sensitivity is measured on the
# classifier's own Validation set instead (see cell 05).
YOLO_DATASET_NAME = "CASHEW_YOLO_MASTER_FINAL_NEG_V03"
YOLO_DATASET_FINGERPRINT = "1acc0edcf7f736f9754e4977b61133a7ec5582b44d8cf3eef27208f49a202add"
YOLO_DATASET_FILE_ID = "1cRQ-kR1lsEvyqvs0Qn8rhUS4mvdi4nD2"
YOLO_DATASET_OVERRIDE = None
YOLO_DATASET_ARCHIVE_NAME = "CASHEW_YOLO_MASTER_FINAL_NEG_V03 (2).zip"
YOLO_DATASET_EXPECTED_BYTES = 246419660

# The classifier's images: the same two archives EXP-029 used.
DATASET_ARCHIVE_NAME = "CashewData_Split (13).zip"
DATASET_ARCHIVE_FILE_ID = "1HUE5ObO1vW0mP2QjgsdObMuW4RVPzNk7"
DATASET_ARCHIVE_EXPECTED_BYTES = 1470301553
REAL_HOLDOUT_ARCHIVE_NAME = "RL_Cashew_holdout (9).zip"
REAL_HOLDOUT_ARCHIVE_FILE_ID = "1QW2JYx8K6zkTs9y34f39OTBlxUOjWosZ"
REAL_HOLDOUT_EXPECTED_BYTES = 370947341

CLASS_NAMES = ["anthracnose", "healthy", "leaf_miner", "red_rust", "not_cashew_leaf"]
DETECTOR_CLASSES = ["anthracnose", "leaf_miner", "red_rust"]  # healthy/not_cashew carry 0 boxes
SEEDS = [42, 123, 2026, 3407, 7777]

# ------------------------------------------------------------
# Gate definition
# ------------------------------------------------------------
# Only anthracnose is gated by default: it is the class the error analysis
# shows as the dumping ground. Widening this to all three disease classes is a
# SECOND factor; run it as EXP-032 rather than folding it in here.
GATED_CLASSES = ["anthracnose"]

# TAU search grid. The calibration cell picks one value on Validation only.
TAU_GRID = [round(0.05 * i, 2) for i in range(1, 19)]   # 0.05 .. 0.90

# Reference numbers measured from the committed FULL ZIP's confusion matrix,
# whose column totals reconcile exactly with the NEG V03 manifest
# (anthracnose 14,507 / leaf_miner 568 / red_rust 32,183 Validation boxes).
# Box-level miss rate, i.e. ground-truth boxes predicted as background:
#   anthracnose 1,125/14,507 = 7.75%   -> image-level sensitivity >= 92.25%
#   leaf_miner      6/   568 = 1.06%
#   red_rust    2,275/32,183 = 7.07%
# The 92.25% figure is a worst case: it assumes every missed box lands on a
# different image. Real sensitivity should be higher because the misses
# scatter across images that carry tens of boxes each.
# Counterweight: the same matrix carries 79,529 false-positive anthracnose
# boxes at the mAP operating point (conf ~ 0.001), so nearly every image has
# SOME anthracnose box. The gate therefore keys on max_conf >= TAU, never on
# "a box exists", and TAU has to be high enough to clear that noise floor.
DETECTOR_BOX_MISS_RATE_REFERENCE = {"anthracnose": 0.0775, "leaf_miner": 0.0106,
                                    "red_rust": 0.0707}
YOLO_INFERENCE_CONF = 0.05   # keep low; TAU does the thresholding afterwards
YOLO_INFERENCE_IOU = 0.7
YOLO_IMGSZ = 640

# ------------------------------------------------------------
# Run flags, enforced in order
# ------------------------------------------------------------
AUTO_STAGE_MISSING_INPUTS = True  # download verified source ZIPs when image folders are absent
DATASET_ARCHIVE_OVERRIDE = None
REAL_HOLDOUT_ARCHIVE_OVERRIDE = None
RUN_LEAKAGE_AUDIT = True
RUN_YOLO_INFERENCE = True
RUN_CALIBRATION = True
RUN_FINAL_EVALUATION = True   # touches Holdout 09; requires the three above

# Hard stop: if any Holdout 09 image is found in the YOLO training split the
# gate is contaminated and the experiment must not proceed.
ABORT_ON_HOLDOUT_LEAKAGE = True


In [3]:
# 01a. FROZEN EXP-029 PREDICTION BUNDLE
# The 15 real EXP-029 prediction CSVs, extracted from its ANALYSIS ZIP.
# Source ANALYSIS ZIP SHA256: 223298083582e13b9312a8f17c9e1e1282e68af45ce33d23db6e2f448639a86e
EXP029_PREDICTION_BUNDLE_SHA256 = '5b091e335e3194dfaeacfa9a2891d6001bb929b8eb1b432e64ecd8a01634952d'
EXP029_PREDICTION_BUNDLE_B64 = '...'


In [4]:
# ============================================================
# 01b. RESOLVE PATHS
# ============================================================
# Point these at the extracted folders. On Colab, download the archives the
# same way EXP-028/029 did and extract them under /content.

PROJECT_ROOT = Path("/content") if Path("/content").is_dir() else Path.cwd()
OUT_DIR = PROJECT_ROOT / "training_results" / "current_dataset" / EXPERIMENT_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

# EXP-029: use an extracted folder or ZIP override, otherwise stage the
# verified 15-table prediction bundle included in this notebook.
import base64, io
_expected_029_members = [
    f"{BASELINE_EXPERIMENT_ID}/seed_{seed}/{split}/predictions.csv"
    for split in ("validation", "test") for seed in SEEDS
] + [f"{BASELINE_EXPERIMENT_ID}/real_holdout/seed_{seed}/predictions.csv"
     for seed in SEEDS]
_exp029_stage_root = PROJECT_ROOT / "exp029"
ANALYSIS_ROOT = (_exp029_stage_root / BASELINE_EXPERIMENT_ID)
_analysis_source = "already extracted"
_archive = None
if CLASSIFIER_ANALYSIS_OVERRIDE:
    _override = Path(CLASSIFIER_ANALYSIS_OVERRIDE).expanduser()
    if _override.is_dir():
        ANALYSIS_ROOT = _override
        _analysis_source = "configured extracted folder"
    elif _override.is_file() and _override.suffix.lower() == ".zip":
        _analysis_source = f"configured ZIP: {_override}"
        _archive = zipfile.ZipFile(_override)
    else:
        raise FileNotFoundError(f"CLASSIFIER_ANALYSIS_OVERRIDE is not a folder or ZIP: {_override}")
elif not all((ANALYSIS_ROOT / name.removeprefix(BASELINE_EXPERIMENT_ID + "/")).is_file()
             for name in _expected_029_members):
    _bundle_bytes = base64.b64decode(EXP029_PREDICTION_BUNDLE_B64, validate=True)
    if hashlib.sha256(_bundle_bytes).hexdigest() != EXP029_PREDICTION_BUNDLE_SHA256:
        raise ValueError("Embedded EXP-029 prediction bundle SHA256 mismatch")
    _archive = zipfile.ZipFile(io.BytesIO(_bundle_bytes))
    _analysis_source = "embedded real EXP-029 ANALYSIS predictions"
if _archive is not None:
    with _archive as _source:
        if _source.testzip() is not None:
            raise ValueError("EXP-029 prediction archive is corrupt")
        _manifest = (json.loads(_source.read("manifest.json"))
                     if "manifest.json" in _source.namelist() else None)
        for _member in _expected_029_members:
            _data = _source.read(_member)
            if _manifest and hashlib.sha256(_data).hexdigest() != _manifest["members"][_member]:
                raise ValueError(f"EXP-029 prediction SHA256 mismatch: {_member}")
            _target = _exp029_stage_root / _member
            if not _target.resolve().is_relative_to(_exp029_stage_root.resolve()):
                raise ValueError(f"Unsafe EXP-029 archive member: {_member}")
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_data)
_missing_029 = [str(ANALYSIS_ROOT / name.removeprefix(BASELINE_EXPERIMENT_ID + "/"))
                for name in _expected_029_members
                if not (ANALYSIS_ROOT / name.removeprefix(BASELINE_EXPERIMENT_ID + "/")).is_file()]
if _missing_029:
    raise FileNotFoundError("EXP-029 needs 15 prediction tables; missing:\n" +
                            "\n".join(_missing_029[:5]) +
                            "\nSet CLASSIFIER_ANALYSIS_OVERRIDE to its ANALYSIS ZIP or extracted folder.")
print("EXP-029 prediction source:", _analysis_source, "| files:", len(_expected_029_members))

# Classifier image folders (Split 13 + Holdout 9):
SPLIT13_VAL_DIR  = PROJECT_ROOT / "dataset_split" / "val"
SPLIT13_TEST_DIR = PROJECT_ROOT / "dataset_split" / "test"
HOLDOUT_IMAGE_DIR = PROJECT_ROOT / "holdout09"

# Detector training images, for the leakage audit:
YOLO_TRAIN_IMAGE_DIR = PROJECT_ROOT / "yolo_neg_v03" / "images" / "train"
# Optional cross-check only; TAU does not depend on these (see cell 05).
YOLO_VAL_IMAGE_DIR = PROJECT_ROOT / "yolo_neg_v03" / "images" / "val"
YOLO_VAL_LABEL_DIR = PROJECT_ROOT / "yolo_neg_v03" / "labels" / "val"
if not Path(YOLO_VAL_IMAGE_DIR).is_dir():
    YOLO_VAL_IMAGE_DIR = YOLO_VAL_LABEL_DIR = None

"""Retrieve and verify the frozen YOLO26s NEG V03 001 checkpoint for EXP-031."""


def ensure_detector_weights(project_root, override, archive_name, archive_url,
                            archive_bytes, archive_sha256, weights_member,
                            weights_sha256):
    """Return a verified best.pt, reusing local files before downloading."""
    project_root = Path(project_root)
    weights = Path(override).expanduser() if override else project_root / 'detector' / 'best.pt'
    if weights.is_file():
        found = file_sha256(weights)
        if found != weights_sha256:
            raise ValueError(f'YOLO best.pt SHA256 mismatch: {found}')
        print('Verified YOLO best.pt:', weights, found[:16])
        return weights
    if override:
        raise FileNotFoundError(f'DETECTOR_WEIGHTS_OVERRIDE does not exist: {weights}')

    candidates = sorted(project_root.rglob(archive_name))
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple YOLO FULL ZIPs found; select one explicitly: {candidates}')
    if candidates:
        archive_path = candidates[0]
        source = 'local archive'
    else:
        import urllib.request
        archive_path = project_root / 'detector' / archive_name
        archive_path.parent.mkdir(parents=True, exist_ok=True)
        partial = archive_path.with_suffix('.zip.part')
        request = urllib.request.Request(archive_url,
                                         headers={'User-Agent': 'EXP-031-checkpoint-provenance'})
        try:
            with urllib.request.urlopen(request, timeout=60) as response, partial.open('wb') as target:
                while True:
                    block = response.read(8 << 20)
                    if not block:
                        break
                    target.write(block)
            if partial.stat().st_size != archive_bytes:
                raise ValueError(f'Incomplete YOLO FULL ZIP: {partial.stat().st_size} bytes')
            actual = file_sha256(partial)
            if actual != archive_sha256:
                raise ValueError(f'YOLO FULL ZIP SHA256 mismatch: {actual}')
            partial.replace(archive_path)
        except Exception as exc:
            partial.unlink(missing_ok=True)
            raise RuntimeError(
                'Could not fetch the pinned YOLO FULL ZIP from GitHub. '
                'Place the verified ZIP on the runtime or set DETECTOR_WEIGHTS_OVERRIDE.'
            ) from exc
        source = 'pinned GitHub commit'

    if archive_path.stat().st_size != archive_bytes:
        raise ValueError(f'YOLO FULL ZIP size mismatch: {archive_path}')
    found_zip = file_sha256(archive_path)
    if found_zip != archive_sha256:
        raise ValueError(f'YOLO FULL ZIP SHA256 mismatch: {found_zip}')
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise ValueError(f'Corrupt YOLO FULL ZIP: {archive_path}')
        checkpoint = archive.read(weights_member)
    found_weights = hashlib.sha256(checkpoint).hexdigest()
    if found_weights != weights_sha256:
        raise ValueError(f'YOLO best.pt SHA256 mismatch: {found_weights}')
    weights.parent.mkdir(parents=True, exist_ok=True)
    temp_weights = weights.with_suffix('.pt.part')
    temp_weights.write_bytes(checkpoint)
    temp_weights.replace(weights)
    print('Verified YOLO best.pt from', source, '|', archive_path,
          '| SHA256', found_weights[:16])
    return weights

DETECTOR_WEIGHTS_PATH = ensure_detector_weights(
    PROJECT_ROOT, DETECTOR_WEIGHTS_OVERRIDE, DETECTOR_FULL_ZIP,
    DETECTOR_FULL_ZIP_URL, DETECTOR_FULL_ZIP_EXPECTED_BYTES,
    DETECTOR_FULL_ZIP_SHA256, DETECTOR_WEIGHTS_MEMBER, DETECTOR_WEIGHTS_SHA256)

for label, path in (("EXP-029 ANALYSIS", ANALYSIS_ROOT),
                    ("Split13 val", SPLIT13_VAL_DIR), ("Split13 test", SPLIT13_TEST_DIR),
                    ("Holdout 09", HOLDOUT_IMAGE_DIR),
                    ("YOLO train images", YOLO_TRAIN_IMAGE_DIR),
                    ("detector best.pt", DETECTOR_WEIGHTS_PATH)):
    print(f"  {'OK  ' if path.exists() else 'MISS'} {label:<20} {path}")
print("\nOutput:", OUT_DIR)


EXP-029 prediction source: already extracted | files: 15
Verified YOLO best.pt: /content/detector/best.pt 66717b5523ba3944
  OK   EXP-029 ANALYSIS     /content/exp029/EXP-DENSENET121-TRANSFER-5SEEDS-029-256px-9c23ac05
  MISS Split13 val          /content/dataset_split/val
  MISS Split13 test         /content/dataset_split/test
  MISS Holdout 09           /content/holdout09
  MISS YOLO train images    /content/yolo_neg_v03/images/train
  OK   detector best.pt     /content/detector/best.pt

Output: /content/training_results/current_dataset/EXP-DENSENET121-YOLOGATE-031


## Load the frozen EXP-029 probabilities

No model is rebuilt. The cell reproduces the EXP-029 headline numbers from the saved softmax vectors and refuses to continue if they do not match.

In [5]:
# ============================================================
# 02. LOAD FROZEN EXP-029 PREDICTIONS
# ============================================================
# No model is rebuilt here. EXP-029 already wrote full softmax vectors for
# Validation, Test and Holdout 09 for all five seeds; this cell normalises the
# two different column schemas it used into one table.

PROB_COLUMNS = [f"prob_{name}" for name in CLASS_NAMES]
CLASS_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}


def normalise_predictions(frame, split, seed):
    """EXP-029 wrote holdout with probability_*/class and val/test with prob_*/true_label."""
    out = pd.DataFrame(index=frame.index)
    if "relative_path" in frame.columns:
        out["image_key"] = frame["relative_path"].astype(str)
    elif "filename" in frame.columns:
        out["image_key"] = frame["filename"].astype(str).map(lambda p: Path(p).as_posix())
    else:
        raise KeyError(f"{split}/{seed}: no relative_path or filename column")
    out["true_label"] = frame["class"] if "class" in frame.columns else frame["true_label"]
    for name in CLASS_NAMES:
        source = f"probability_{name}" if f"probability_{name}" in frame.columns else f"prob_{name}"
        if source not in frame.columns:
            raise KeyError(f"{split}/{seed}: missing probability column for {name}")
        out[f"prob_{name}"] = frame[source].astype(float)
    out["split"] = split
    out["seed"] = seed
    out["basename"] = out["image_key"].map(lambda p: Path(p).name)
    # EXP-029 paths end in class/image.jpg for all three splits. The absolute
    # Colab prefix in Val/Test is not stable across runtimes.
    out["image_relpath"] = out["image_key"].map(
        lambda p: "/".join(Path(p).parts[-2:]))
    if not (out["image_relpath"].str.split("/").str[0] == out["true_label"]).all():
        raise ValueError(f"{split}/{seed}: image path class differs from true_label")
    out["evidence_key"] = out["split"] + "/" + out["image_relpath"]
    if out["evidence_key"].duplicated().any():
        raise ValueError(f"{split}/{seed}: duplicate split/class/image prediction key")
    probabilities = out[PROB_COLUMNS].to_numpy()
    row_sums = probabilities.sum(axis=1)
    if not np.allclose(row_sums, 1.0, atol=1e-4):
        raise ValueError(f"{split}/{seed}: softmax rows do not sum to one")
    out["baseline_pred"] = [CLASS_NAMES[i] for i in probabilities.argmax(axis=1)]
    return out


SPLIT_SOURCES = {
    "validation": lambda seed: ANALYSIS_ROOT / f"seed_{seed}" / "validation" / "predictions.csv",
    "test": lambda seed: ANALYSIS_ROOT / f"seed_{seed}" / "test" / "predictions.csv",
    "holdout": lambda seed: ANALYSIS_ROOT / "real_holdout" / f"seed_{seed}" / "predictions.csv",
}

frames = []
for split, locate in SPLIT_SOURCES.items():
    for seed in SEEDS:
        path = locate(seed)
        if not path.is_file():
            raise FileNotFoundError(f"EXP-029 predictions missing: {path}")
        frames.append(normalise_predictions(pd.read_csv(path), split, seed))
predictions_df = pd.concat(frames, ignore_index=True)

# Every seed must cover the identical image list for a split, otherwise the
# paired comparison below is not actually paired.
for split, group in predictions_df.groupby("split"):
    per_seed = group.groupby("seed")["evidence_key"].apply(lambda s: tuple(sorted(s)))
    if per_seed.nunique() != 1:
        raise ValueError(f"{split}: the five seeds do not share one image list")
    print(f"  {split:<11}: {len(per_seed.iloc[0]):5d} images x {len(SEEDS)} seeds")

baseline_accuracy = (predictions_df.groupby(["split", "seed"])
                     .apply(lambda g: float((g["baseline_pred"] == g["true_label"]).mean()),
                            include_groups=False)
                     .rename("accuracy").reset_index())
print("\nEXP-029 baseline reproduced from saved probabilities:")
for split in ("validation", "test", "holdout"):
    values = baseline_accuracy[baseline_accuracy["split"] == split]["accuracy"]
    print(f"  {split:<11}: {100*values.mean():.2f} +/- {100*values.std(ddof=1):.2f}%")

_test = baseline_accuracy[baseline_accuracy["split"] == "test"]["accuracy"].mean() * 100
_hold = baseline_accuracy[baseline_accuracy["split"] == "holdout"]["accuracy"].mean() * 100
print(f"\n  Baseline gap Test - Holdout: {_test - _hold:.2f} points "
      f"(EXP-029 reported 13.54; a mismatch here means the wrong ZIP is loaded)")
if abs((_test - _hold) - 13.54) > 0.2:
    raise ValueError("Loaded probabilities do not reproduce the EXP-029 gap; check the ANALYSIS ZIP")


  holdout    :   126 images x 5 seeds
  test       :   685 images x 5 seeds
  validation :  1452 images x 5 seeds

EXP-029 baseline reproduced from saved probabilities:
  validation : 97.37 +/- 0.20%
  test       : 96.23 +/- 0.49%
  holdout    : 82.70 +/- 1.30%

  Baseline gap Test - Holdout: 13.54 points (EXP-029 reported 13.54; a mismatch here means the wrong ZIP is loaded)


## Leakage audit

The gate is only honest if the detector has never seen the images it vouches for. Exact match uses the SHA256 of decoded RGB pixels, which survives re-encoding; near match uses a 64-bit pHash at Hamming <= 4.

## Prepare image inputs for leakage audit

If the original Split 13, Holdout 09 or YOLO NEG V03 image folders are missing, this cell downloads the exact source ZIPs from Drive, verifies archive size and checksum, extracts safely, and matches classifier filenames to the frozen EXP-029 predictions. Set the three `*_OVERRIDE` paths in CONFIG to use local ZIPs.


In [6]:
"""Stage the original EXP-029 and YOLO NEG V03 image archives for EXP-031."""

import stat
from collections import Counter

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
STAGE_ROOT = PROJECT_ROOT / 'exp031_inputs'

def image_files(root):
    return sorted(path for path in Path(root).rglob('*')
                  if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)

def archive_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

def source_archive(file_id, name, expected_bytes, override):
    if override:
        path = Path(override).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f'Archive override missing: {path}')
    else:
        path = STAGE_ROOT / 'archives' / name
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.is_file() or path.stat().st_size != expected_bytes or not zipfile.is_zipfile(path):
            if not file_id:
                raise FileNotFoundError(f'{name}: set a Drive file ID or archive override')
            import google.auth
            from googleapiclient.discovery import build
            from googleapiclient.http import MediaIoBaseDownload
            if Path('/content').is_dir():
                from google.colab import auth
                auth.authenticate_user()
            credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
            service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
            meta = service.files().get(fileId=file_id, fields='id,name,size,md5Checksum',
                                       supportsAllDrives=True).execute()
            if meta['name'] != name or int(meta['size']) != expected_bytes:
                raise ValueError(f'Drive source differs from expected {name}: {meta}')
            partial = path.with_suffix(path.suffix + '.part')
            request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
            with partial.open('wb') as stream:
                downloader = MediaIoBaseDownload(stream, request, chunksize=8 << 20)
                done = False
                while not done:
                    progress, done = downloader.next_chunk(num_retries=3)
                    if progress:
                        print(name, f'{progress.progress():.0%}')
            if partial.stat().st_size != expected_bytes or not zipfile.is_zipfile(partial):
                raise RuntimeError(f'Incomplete archive download: {name}')
            if meta.get('md5Checksum'):
                digest = hashlib.md5()
                with partial.open('rb') as stream:
                    for block in iter(lambda: stream.read(8 << 20), b''):
                        digest.update(block)
                if digest.hexdigest() != meta['md5Checksum']:
                    raise RuntimeError(f'Drive MD5 mismatch: {name}')
            partial.replace(path)
    if path.stat().st_size != expected_bytes or not zipfile.is_zipfile(path):
        raise ValueError(f'Archive size/type mismatch: {path}')
    return path

def extract_verified(archive_path, destination):
    archive_path, destination = Path(archive_path), Path(destination)
    checksum = archive_sha256(archive_path)
    marker = destination / '.archive_sha256'
    if destination.is_dir():
        if not marker.is_file() or marker.read_text(encoding='utf-8').strip() != checksum:
            raise RuntimeError(f'Unverified/incomplete extraction: {destination}')
        return destination
    staging = destination.with_name(destination.name + '_staging')
    if staging.exists():
        raise RuntimeError(f'Incomplete staging directory: {staging}')
    staging.mkdir(parents=True)
    with zipfile.ZipFile(archive_path) as archive:
        resolved_root = staging.resolve()
        for member in archive.infolist():
            target = (staging / member.filename.replace('\\', '/')).resolve()
            if (stat.S_ISLNK(member.external_attr >> 16) or
                    not target.is_relative_to(resolved_root)):
                raise ValueError(f'Unsafe archive member: {member.filename}')
        archive.extractall(staging)
    (staging / '.archive_sha256').write_text(checksum + '\n', encoding='utf-8')
    staging.rename(destination)
    return destination

def unique_root(extracted, kind):
    candidates = [extracted, *[p for p in extracted.rglob('*') if p.is_dir()]]
    if kind == 'split13':
        matches = [p for p in candidates if (p / 'val').is_dir() and (p / 'test').is_dir()]
    elif kind == 'holdout09':
        holdout_classes = set(predictions_df.loc[predictions_df['split'] == 'holdout', 'true_label'])
        matches = [p for p in candidates if
                   {q.name for q in p.iterdir() if q.is_dir()} == holdout_classes]
    elif kind == 'yolo':
        matches = [p for p in candidates if (p / 'images' / 'train').is_dir() and
                   (p / 'labels' / 'train').is_dir()]
    else:
        raise ValueError(kind)
    if len(matches) != 1:
        raise RuntimeError(f'{kind}: expected one dataset root, found {matches}')
    return matches[0]

if not DETECTOR_WEIGHTS_PATH.is_file():
    raise FileNotFoundError(f'YOLO best.pt missing: {DETECTOR_WEIGHTS_PATH}; '
                            'set DETECTOR_WEIGHTS_OVERRIDE or place the verified FULL ZIP on this runtime')

if AUTO_STAGE_MISSING_INPUTS:
    if not Path(SPLIT13_VAL_DIR).is_dir() or not Path(SPLIT13_TEST_DIR).is_dir():
        archive = source_archive(DATASET_ARCHIVE_FILE_ID, DATASET_ARCHIVE_NAME,
                                 DATASET_ARCHIVE_EXPECTED_BYTES, DATASET_ARCHIVE_OVERRIDE)
        root = unique_root(extract_verified(archive, STAGE_ROOT / 'split13'), 'split13')
        SPLIT13_VAL_DIR, SPLIT13_TEST_DIR = root / 'val', root / 'test'
    if not Path(HOLDOUT_IMAGE_DIR).is_dir():
        archive = source_archive(REAL_HOLDOUT_ARCHIVE_FILE_ID, REAL_HOLDOUT_ARCHIVE_NAME,
                                 REAL_HOLDOUT_EXPECTED_BYTES, REAL_HOLDOUT_ARCHIVE_OVERRIDE)
        HOLDOUT_IMAGE_DIR = unique_root(
            extract_verified(archive, STAGE_ROOT / 'holdout09'), 'holdout09')
    if not Path(YOLO_TRAIN_IMAGE_DIR).is_dir():
        archive = source_archive(YOLO_DATASET_FILE_ID, YOLO_DATASET_ARCHIVE_NAME,
                                 YOLO_DATASET_EXPECTED_BYTES, YOLO_DATASET_OVERRIDE)
        yolo_root = unique_root(extract_verified(archive, STAGE_ROOT / 'yolo'), 'yolo')
        YOLO_TRAIN_IMAGE_DIR = yolo_root / 'images' / 'train'
        YOLO_VAL_IMAGE_DIR = yolo_root / 'images' / 'val'
        YOLO_VAL_LABEL_DIR = yolo_root / 'labels' / 'val'

for split, folder, expected_count in (
        ('validation', SPLIT13_VAL_DIR, 1452),
        ('test', SPLIT13_TEST_DIR, 685),
        ('holdout', HOLDOUT_IMAGE_DIR, 126)):
    files = image_files(folder) if Path(folder).is_dir() else []
    if len(files) != expected_count:
        raise FileNotFoundError(f'{split}: expected {expected_count} images under {folder}; found {len(files)}')
    expected_paths = Counter(predictions_df.loc[
        (predictions_df['split'] == split) & (predictions_df['seed'] == SEEDS[0]), 'image_relpath'])
    actual_paths = Counter(path.relative_to(folder).as_posix() for path in files)
    if actual_paths != expected_paths:
        raise ValueError(f'{split}: relative image paths do not match frozen EXP-029 predictions')
    print(f'  staged {split}: {len(files)} images at {folder}')
train_images = image_files(YOLO_TRAIN_IMAGE_DIR) if Path(YOLO_TRAIN_IMAGE_DIR).is_dir() else []
if not train_images:
    raise FileNotFoundError(f'YOLO training images missing: {YOLO_TRAIN_IMAGE_DIR}')
print(f'  staged YOLO train: {len(train_images)} images at {YOLO_TRAIN_IMAGE_DIR}')


  staged validation: 1452 images at /content/exp031_inputs/split13/CashewData_Split/val
  staged test: 685 images at /content/exp031_inputs/split13/CashewData_Split/test
  staged holdout: 126 images at /content/exp031_inputs/holdout09/RL_Cashew_holdout
  staged YOLO train: 3497 images at /content/exp031_inputs/yolo/CASHEW_YOLO_MASTER_FINAL_NEG_V03/images/train


In [7]:
# ============================================================
# 03. LEAKAGE AUDIT — YOLO training split vs the classifier's images
# ============================================================
# The gate is only honest if the detector has never seen the images it is
# about to vouch for. The detector was trained on CASHEW_YOLO_MASTER_FINAL_
# NEG_V03, which was assembled separately from Split (13), so overlap is
# plausible and must be measured rather than assumed.
#
# Two levels, matching the audit style already used in EXP-028:
#   exact : SHA256 of decoded RGB pixels (survives re-encoding, unlike file bytes)
#   near  : 64-bit perceptual hash, Hamming distance <= 4

if RUN_LEAKAGE_AUDIT:
    def pixel_sha256(path):
        with Image.open(path) as handle:
            rgb = ImageOps.exif_transpose(handle).convert("RGB")
            digest = hashlib.sha256()
            digest.update(f"{rgb.width}x{rgb.height}:".encode())
            digest.update(rgb.tobytes())
            return digest.hexdigest()

    def phash64(path):
        with Image.open(path) as handle:
            grey = ImageOps.exif_transpose(handle).convert("L").resize((32, 32), Image.BILINEAR)
        pixels = np.asarray(grey, dtype=np.float32)
        frequencies = np.fft.fft2(pixels)[:8, :8].real
        flat = frequencies.flatten()[1:]
        bits = flat > np.median(flat)
        return int("".join("1" if bit else "0" for bit in bits), 2)

    IMAGE_INPUT_HINTS = {
        "yolo_train": "YOLO_TRAIN_IMAGE_DIR; extract NEG V03 dataset ZIP (Drive ID 1cRQ-kR1lsEvyqvs0Qn8rhUS4mvdi4nD2) and select its images/train folder",
        "classifier_holdout09": "HOLDOUT_IMAGE_DIR; extract RL_Cashew_holdout (9).zip (Drive ID 1QW2JYx8K6zkTs9y34f39OTBlxUOjWosZ)",
        "classifier_validation": "SPLIT13_VAL_DIR; extract CashewData_Split (13).zip (Drive ID 1HUE5ObO1vW0mP2QjgsdObMuW4RVPzNk7) and select val",
        "classifier_test": "SPLIT13_TEST_DIR; select test from the same Split 13 archive",
    }

    def image_paths(root, label):
        root = Path(root)
        hint = IMAGE_INPUT_HINTS[label]
        if not root.is_dir():
            raise FileNotFoundError(f"{label}: image directory does not exist: {root}\nSet {hint}")
        paths = sorted(path for path in root.rglob("*")
                       if path.is_file() and path.suffix.lower() in
                       {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
        if not paths:
            raise FileNotFoundError(f"{label}: no images found under {root}\nSet {hint}")
        return paths

    def fingerprint_folder(root, label):
        rows = []
        split = {"classifier_validation": "validation", "classifier_test": "test",
                 "classifier_holdout09": "holdout", "yolo_train": "yolo_train"}[label]
        for path in image_paths(root, label):
            relpath = path.relative_to(root).as_posix()
            rows.append({"source": label, "path": str(path), "basename": path.name,
                         "image_relpath": relpath, "evidence_key": f"{split}/{relpath}",
                         "pixel_sha256": pixel_sha256(path), "phash64": phash64(path)})
        frame = pd.DataFrame(rows, columns=["source", "path", "basename", "image_relpath",
                                           "evidence_key", "pixel_sha256", "phash64"])
        print(f"  fingerprinted {label:<24} {len(frame):5d} images")
        return frame

    yolo_train_fp = fingerprint_folder(YOLO_TRAIN_IMAGE_DIR, "yolo_train")
    holdout_fp = fingerprint_folder(HOLDOUT_IMAGE_DIR, "classifier_holdout09")
    val_fp = fingerprint_folder(SPLIT13_VAL_DIR, "classifier_validation")
    test_fp = fingerprint_folder(SPLIT13_TEST_DIR, "classifier_test")

    def compare(target_fp, name):
        exact = target_fp.merge(yolo_train_fp, on="pixel_sha256", suffixes=("", "_yolo"))
        yolo_hashes = yolo_train_fp["phash64"].to_numpy()
        near_rows = []
        for _, row in target_fp.iterrows():
            distances = np.array([bin(int(row["phash64"]) ^ int(h)).count("1") for h in yolo_hashes])
            hits = np.flatnonzero(distances <= 4)
            for index in hits:
                near_rows.append({"target": row["path"],
                                  "target_evidence_key": row["evidence_key"],
                                  "target_basename": row["basename"],
                                  "yolo_train": yolo_train_fp.iloc[index]["path"],
                                  "phash_hamming": int(distances[index])})
        near = pd.DataFrame(near_rows)
        print(f"  {name:<24} exact pixel overlap: {len(exact):3d} | near-duplicate pairs: {len(near):3d}")
        return exact, near

    print("\nComparing each classifier split against the YOLO training images:")
    holdout_exact, holdout_near = compare(holdout_fp, "Holdout 09")
    val_exact, val_near = compare(val_fp, "Validation (Split 13)")
    test_exact, test_near = compare(test_fp, "Test (Split 13)")

    for frame, filename in ((holdout_exact, "leak_holdout_exact.csv"),
                            (holdout_near, "leak_holdout_near.csv"),
                            (val_exact, "leak_validation_exact.csv"),
                            (val_near, "leak_validation_near.csv"),
                            (test_exact, "leak_test_exact.csv"),
                            (test_near, "leak_test_near.csv")):
        frame.to_csv(OUT_DIR / filename, index=False)

    # Validation images the detector has seen are removed from TAU calibration.
    CALIBRATION_EXCLUDED = (set(val_exact["evidence_key"]) |
                            set(val_near.get("target_evidence_key", pd.Series(dtype=str))))
    print(f"\n  Validation images excluded from TAU calibration: {len(CALIBRATION_EXCLUDED)}")

    leakage_report = {
        "detector": DETECTOR_NAME,
        "yolo_dataset": YOLO_DATASET_NAME,
        "yolo_dataset_fingerprint": YOLO_DATASET_FINGERPRINT,
        "yolo_train_images": int(len(yolo_train_fp)),
        "holdout_exact_overlap": int(len(holdout_exact)),
        "holdout_near_duplicate_pairs": int(len(holdout_near)),
        "validation_exact_overlap": int(len(val_exact)),
        "validation_near_duplicate_pairs": int(len(val_near)),
        "test_exact_overlap": int(len(test_exact)),
        "test_near_duplicate_pairs": int(len(test_near)),
        "calibration_excluded_images": sorted(CALIBRATION_EXCLUDED),
        "status": "PASS" if len(holdout_exact) == 0 else "FAIL",
    }
    save_json(OUT_DIR / "leakage_audit.json", leakage_report)

    if len(holdout_exact) and ABORT_ON_HOLDOUT_LEAKAGE:
        raise RuntimeError(
            f"{len(holdout_exact)} Holdout 09 images are in the YOLO training split. "
            "The gate would be vouching for images the detector memorised. "
            "Retrain the detector without them, or score the gate on a holdout the "
            "detector has never seen.")
    if len(holdout_near):
        print(f"\n  WARNING: {len(holdout_near)} Holdout 09 near-duplicate pairs with YOLO train. "
              "Not an automatic abort, but review them before publishing the result.")
    print("\nLeakage audit:", leakage_report["status"])
else:
    print("Leakage audit NOT_EXECUTED — the gate must not be calibrated in this state")


  fingerprinted yolo_train                3497 images
  fingerprinted classifier_holdout09       126 images
  fingerprinted classifier_validation     1452 images
  fingerprinted classifier_test            685 images

Comparing each classifier split against the YOLO training images:
  Holdout 09               exact pixel overlap:   0 | near-duplicate pairs:   0
  Validation (Split 13)    exact pixel overlap:   0 | near-duplicate pairs: 247
  Test (Split 13)          exact pixel overlap:   0 | near-duplicate pairs:  62

  Validation images excluded from TAU calibration: 91

Leakage audit: PASS


## Detector inference

Many boxes per image are reduced to a handful of per-image statistics. Boxes are kept at a low confidence here; `TAU` does the thresholding afterwards, so the grid search needs no re-inference.

In [8]:
# ============================================================
# 04. YOLO INFERENCE — many boxes per image, reduced to image-level evidence
# ============================================================
# The detector emits a variable number of lesion boxes per image. The gate
# never looks at an individual box; it reduces them to a few per-image
# statistics. This is what makes "many boxes per image" a feature rather than
# a problem: no crop, no aggregation of per-crop classifications, no
# retraining of the classifier.
#
# Reduction per image and per detector class c:
#   max_conf_c    highest confidence among boxes of class c   (0.0 if none)
#   n_boxes_c     count of boxes of class c above YOLO_INFERENCE_CONF
#   area_frac_c   union-free sum of box areas / image area, clipped at 1.0
# Only max_conf_c drives the default gate; the rest are recorded so EXP-032
# can try a richer rule without re-running inference.

if RUN_YOLO_INFERENCE:
    from ultralytics import YOLO

    detector = YOLO(str(DETECTOR_WEIGHTS_PATH))
    recorded_names = detector.names if isinstance(detector.names, dict) else dict(enumerate(detector.names))
    detector_class_order = [recorded_names[i] for i in sorted(recorded_names)]
    if detector_class_order != DETECTOR_CLASSES:
        raise ValueError(f"Detector class order {detector_class_order} != {DETECTOR_CLASSES}")
    print("Detector classes:", detector_class_order)
    print("Weights SHA256   :", file_sha256(DETECTOR_WEIGHTS_PATH))

    def evidence_for_folder(image_dir, label):
        paths = sorted(p for p in Path(image_dir).rglob("*")
                       if p.is_file() and p.suffix.lower() in
                       {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
        if not paths:
            raise FileNotFoundError(f"No images under {image_dir}")
        rows = []
        started = time.perf_counter()
        for start in range(0, len(paths), 16):
            batch = paths[start:start + 16]
            results = detector.predict([str(p) for p in batch], imgsz=YOLO_IMGSZ,
                                       conf=YOLO_INFERENCE_CONF, iou=YOLO_INFERENCE_IOU,
                                       verbose=False)
            for path, result in zip(batch, results):
                height, width = result.orig_shape
                image_area = float(height * width)
                split = {"classifier_validation": "validation", "classifier_test": "test",
                         "classifier_holdout09": "holdout", "yolo_validation": "yolo_validation"}[label]
                relpath = path.relative_to(image_dir).as_posix()
                row = {"source": label, "image_key": str(path), "basename": path.name,
                       "image_relpath": relpath, "evidence_key": f"{split}/{relpath}"}
                boxes = result.boxes
                classes = (boxes.cls.cpu().numpy().astype(int) if boxes is not None and len(boxes)
                           else np.zeros(0, dtype=int))
                confidences = (boxes.conf.cpu().numpy() if boxes is not None and len(boxes)
                               else np.zeros(0, dtype=float))
                xyxy = (boxes.xyxy.cpu().numpy() if boxes is not None and len(boxes)
                        else np.zeros((0, 4), dtype=float))
                areas = ((xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
                         if len(xyxy) else np.zeros(0, dtype=float))
                for index, name in enumerate(DETECTOR_CLASSES):
                    mask = classes == index
                    row[f"max_conf_{name}"] = float(confidences[mask].max()) if mask.any() else 0.0
                    row[f"n_boxes_{name}"] = int(mask.sum())
                    row[f"area_frac_{name}"] = float(min(areas[mask].sum() / image_area, 1.0)) if mask.any() else 0.0
                row["n_boxes_any"] = int(len(classes))
                row["max_conf_any"] = float(confidences.max()) if len(confidences) else 0.0
                rows.append(row)
        elapsed = time.perf_counter() - started
        frame = pd.DataFrame(rows)
        print(f"  {label:<22} {len(frame):5d} images | {1000*elapsed/len(frame):6.1f} ms/image "
              f"| mean boxes/image {frame['n_boxes_any'].mean():5.1f} "
              f"| images with 0 boxes {int((frame['n_boxes_any'] == 0).sum())}")
        return frame

    print("\nRunning detector (boxes are kept at a low conf; TAU thresholds later):")
    evidence_val = evidence_for_folder(SPLIT13_VAL_DIR, "classifier_validation")
    evidence_test = evidence_for_folder(SPLIT13_TEST_DIR, "classifier_test")
    evidence_holdout = evidence_for_folder(HOLDOUT_IMAGE_DIR, "classifier_holdout09")

    # Optional cross-check against the detector's own validation split, if the
    # YOLO dataset happens to be mounted. TAU does NOT depend on it: cell 05
    # measures image-level sensitivity on the classifier's own Validation,
    # which is the population the gate actually runs on.
    evidence_yolo_val = None
    if YOLO_VAL_IMAGE_DIR is not None and Path(YOLO_VAL_IMAGE_DIR).is_dir():
        evidence_yolo_val = evidence_for_folder(YOLO_VAL_IMAGE_DIR, "yolo_validation")
        gt_rows = []
        for _, row in evidence_yolo_val.iterrows():
            label_path = Path(YOLO_VAL_LABEL_DIR) / Path(row["basename"]).with_suffix(".txt").name
            present = set()
            if label_path.is_file():
                for line in label_path.read_text(encoding="utf-8").splitlines():
                    parts = line.split()
                    if parts:
                        present.add(DETECTOR_CLASSES[int(float(parts[0]))])
            gt_rows.append({"evidence_key": row["evidence_key"],
                            **{f"gt_has_{name}": name in present for name in DETECTOR_CLASSES},
                            "gt_negative": len(present) == 0})
        evidence_yolo_val = evidence_yolo_val.merge(pd.DataFrame(gt_rows), on="evidence_key",
                                                    validate="one_to_one")
        evidence_yolo_val.to_csv(OUT_DIR / "yolo_val_evidence_with_gt.csv", index=False)
        print("  YOLO validation ground truth: "
              f"{int(evidence_yolo_val['gt_negative'].sum())} negative images, "
              + ", ".join(f"{name} {int(evidence_yolo_val['gt_has_' + name].sum())}"
                          for name in DETECTOR_CLASSES))
    else:
        print("  YOLO validation split not mounted — skipping the optional cross-check")

    evidence_df = pd.concat([evidence_val, evidence_test, evidence_holdout], ignore_index=True)
    evidence_df.to_csv(OUT_DIR / "yolo_image_evidence.csv", index=False)
    save_json(OUT_DIR / "detector_provenance.json", {
        "detector": DETECTOR_NAME,
        "weights_sha256": file_sha256(DETECTOR_WEIGHTS_PATH),
        "declared_weights_sha256": DETECTOR_WEIGHTS_SHA256,
        "classes": detector_class_order,
        "imgsz": YOLO_IMGSZ, "conf": YOLO_INFERENCE_CONF, "iou": YOLO_INFERENCE_IOU,
        "ultralytics": __import__("ultralytics").__version__,
        "images_scored": {"validation": int(len(evidence_val)), "test": int(len(evidence_test)),
                          "holdout": int(len(evidence_holdout))},
    })
    print("\nSaved image-level evidence for", len(evidence_df), "images")
else:
    evidence_df = pd.read_csv(OUT_DIR / "yolo_image_evidence.csv")
    yolo_val_cache = OUT_DIR / "yolo_val_evidence_with_gt.csv"
    evidence_yolo_val = pd.read_csv(yolo_val_cache) if yolo_val_cache.is_file() else None
    print("Loaded cached evidence for", len(evidence_df), "images")

# Migrate cached evidence written by an older EXP-031 cell in the same runtime.
# Cross-runtime cache paths are refused; the detector must score those images again.
if "evidence_key" not in evidence_df.columns:
    roots = {"classifier_validation": Path(SPLIT13_VAL_DIR),
             "classifier_test": Path(SPLIT13_TEST_DIR),
             "classifier_holdout09": Path(HOLDOUT_IMAGE_DIR)}
    names = {"classifier_validation": "validation", "classifier_test": "test",
             "classifier_holdout09": "holdout"}
    relpaths, keys = [], []
    for row in evidence_df.itertuples(index=False):
        root = roots[row.source]
        path = Path(row.image_key)
        if not path.is_relative_to(root):
            raise RuntimeError("Cached detector evidence uses different image paths. "
                               "Set RUN_YOLO_INFERENCE=True and rerun this cell.")
        relpath = path.relative_to(root).as_posix()
        relpaths.append(relpath)
        keys.append(f"{names[row.source]}/{relpath}")
    evidence_df["image_relpath"] = relpaths
    evidence_df["evidence_key"] = keys
    evidence_df.to_csv(OUT_DIR / "yolo_image_evidence.csv", index=False)
    print("Migrated cached detector evidence to split-qualified image keys")
if evidence_df["evidence_key"].duplicated().any():
    duplicated = evidence_df.loc[evidence_df["evidence_key"].duplicated(), "evidence_key"].tolist()
    raise ValueError(f"Duplicate detector evidence keys: {duplicated[:5]}")
expected_keys = set(predictions_df["evidence_key"])
actual_keys = set(evidence_df["evidence_key"])
if expected_keys != actual_keys:
    raise ValueError(f"Detector evidence image mismatch: "
                     f"missing={sorted(expected_keys - actual_keys)[:5]}, "
                     f"extra={sorted(actual_keys - expected_keys)[:5]}")
EVIDENCE_BY_KEY = evidence_df.set_index("evidence_key")
print("Verified unique detector evidence for", len(EVIDENCE_BY_KEY), "split-qualified images")


Detector classes: ['anthracnose', 'leaf_miner', 'red_rust']
Weights SHA256   : 66717b5523ba394443495378a326da26f0d24e98a331f98480f79d9d6fa18166

Running detector (boxes are kept at a low conf; TAU thresholds later):
  classifier_validation   1452 images |    4.9 ms/image | mean boxes/image  90.2 | images with 0 boxes 281
  classifier_test          685 images |    4.3 ms/image | mean boxes/image  75.8 | images with 0 boxes 79
  classifier_holdout09     126 images |   50.5 ms/image | mean boxes/image  52.1 | images with 0 boxes 11
  yolo_validation          951 images |    2.5 ms/image | mean boxes/image 122.8 | images with 0 boxes 46
  YOLO validation ground truth: 58 negative images, anthracnose 752, leaf_miner 270, red_rust 455

Saved image-level evidence for 2263 images
Verified unique detector evidence for 2263 split-qualified images


## The gate and TAU calibration

`TAU` is chosen on Validation only. If no `TAU` improves Validation, the cell says so: scoring Holdout after that would be fishing.

In [9]:
# ============================================================
# 05. THE GATE, AND TAU CALIBRATION ON VALIDATION ONLY
# ============================================================

GATED_INDICES = [CLASS_INDEX[name] for name in GATED_CLASSES]
for name in GATED_CLASSES:
    if name not in DETECTOR_CLASSES:
        raise ValueError(f"{name} is gated but the detector cannot emit it")


def apply_gate(frame, tau, gated_classes=None):
    """Veto a gated-class prediction that has no lesion evidence at >= tau.

    frame must carry prob_* columns, baseline_pred, and one max_conf_<class>
    column per gated class. Returns the gated prediction as a numpy array of
    class-name strings. Pure function: no globals beyond CLASS_NAMES.
    """
    gated_classes = GATED_CLASSES if gated_classes is None else gated_classes
    probabilities = frame[PROB_COLUMNS].to_numpy()
    order = np.argsort(-probabilities, axis=1)
    top1 = order[:, 0]
    top2 = order[:, 1]
    final = top1.copy()
    for name in gated_classes:
        index = CLASS_INDEX[name]
        evidence = frame[f"max_conf_{name}"].to_numpy()
        veto = (top1 == index) & (evidence < tau)
        final[veto] = top2[veto]
    return np.array([CLASS_NAMES[i] for i in final]), final != top1


def accuracy_at(frame, tau):
    gated, _ = apply_gate(frame, tau)
    return float((gated == frame["true_label"].to_numpy()).mean())


# Attach the detector evidence to every prediction row.
for name in DETECTOR_CLASSES:
    predictions_df[f"max_conf_{name}"] = predictions_df["evidence_key"].map(
        EVIDENCE_BY_KEY[f"max_conf_{name}"]).astype(float)
    predictions_df[f"n_boxes_{name}"] = predictions_df["evidence_key"].map(
        EVIDENCE_BY_KEY[f"n_boxes_{name}"]).astype("Int64")
missing = predictions_df[[f"max_conf_{n}" for n in DETECTOR_CLASSES]].isna().any(axis=1)
if missing.any():
    raise ValueError(f"{int(missing.sum())} prediction rows have no detector evidence; "
                     "the detector did not score every classifier image")

if RUN_CALIBRATION:
    # ------------------------------------------------------------
    # Why TAU is NOT chosen by maximising classifier Validation accuracy.
    #
    # Validation is 1393 of 1452 Kaggle-style images where the classifier is
    # already ~97% correct, so almost every veto there costs more than it
    # gains and the end-to-end optimum collapses to "never veto". That is the
    # same trap as selecting seeds on overall Validation loss: the metric is
    # dominated by the source domain, and the gate exists for the target one.
    #
    # Instead TAU is read off the DETECTOR's own operating characteristic,
    # measured on the detector's own validation split with its YOLO ground
    # truth. The classifier's Validation is kept only as a safety constraint:
    # whatever TAU we pick must not cost more than MAX_VALIDATION_COST points
    # in-distribution. Holdout 09 is never consulted.
    # ------------------------------------------------------------
    # TAU is chosen by Youden's J = sensitivity - false_alarm, the textbook
    # operating point of a diagnostic test. J is a property of the detector
    # alone and does not depend on class base rates, which is exactly what is
    # needed here: the base rate of wrong anthracnose predictions is low in
    # Validation and high in the field, and a criterion that depended on it
    # would once again be dominated by the source domain.
    #
    # The two budgets below are guard rails, not objectives. SENSITIVITY_FLOOR
    # stops the gate from gutting the anthracnose class; MAX_VALIDATION_COST
    # stops it from doing real in-distribution damage. They are deliberately
    # loose: an early version of this cell used 0.95 / 0.5 and the real
    # detector's false-positive floor (79,529 spurious anthracnose boxes at the
    # mAP operating point) made only TAU = 0.05 admissible, where the gate has
    # false_alarm 0.79 and is useless.
    SENSITIVITY_FLOOR = 0.85      # never veto true-lesion images more than this often
    MAX_VALIDATION_COST = 2.00    # points of in-distribution accuracy we accept losing

    # Safety constraint and operating characteristic both come from the
    # classifier's Validation, with detector-seen images dropped.
    calibration = predictions_df[(predictions_df["split"] == "validation") &
                                 (~predictions_df["evidence_key"].isin(CALIBRATION_EXCLUDED))].copy()

    # Image-level sensitivity is measured on exactly the population the gate
    # runs on: one row per unique Validation image, labelled by the
    # classifier's own 5-class ground truth. This needs no YOLO dataset, and
    # unlike the detector's box-level recall it answers the only question the
    # gate asks -- "does this image show at least one lesion of class c at
    # confidence >= TAU?".
    unique_images = calibration.drop_duplicates("evidence_key")[
        ["evidence_key", "true_label"] + [f"max_conf_{n}" for n in DETECTOR_CLASSES]]
    print(f"Image-level operating characteristic on {len(unique_images)} unique Validation images")
    for name in GATED_CLASSES:
        print(f"  true {name}: {int((unique_images['true_label'] == name).sum())} images | "
              f"other: {int((unique_images['true_label'] != name).sum())}")

    detector_rows = []
    for tau in TAU_GRID:
        row = {"tau": tau}
        for name in GATED_CLASSES:
            evidence = unique_images[f"max_conf_{name}"].to_numpy()
            positive = (unique_images["true_label"] == name).to_numpy()
            fires = evidence >= tau
            row[f"sensitivity_{name}"] = float(fires[positive].mean()) if positive.any() else float("nan")
            row[f"false_alarm_{name}"] = float(fires[~positive].mean()) if (~positive).any() else float("nan")
            row[f"correct_veto_rate_{name}"] = float((~fires)[~positive].mean()) if (~positive).any() else float("nan")
        detector_rows.append(row)
    detector_df = pd.DataFrame(detector_rows)
    baseline_val = float(np.mean([
        (calibration[calibration["seed"] == s]["baseline_pred"] ==
         calibration[calibration["seed"] == s]["true_label"]).mean() for s in SEEDS]))
    detector_df["val_accuracy_mean"] = [
        float(np.mean([accuracy_at(calibration[calibration["seed"] == s], tau) for s in SEEDS]))
        for tau in TAU_GRID]
    detector_df["val_cost_points"] = (baseline_val - detector_df["val_accuracy_mean"]) * 100
    detector_df.to_csv(OUT_DIR / "tau_calibration.csv", index=False)

    print(f"Classifier Validation baseline (ungated): {100*baseline_val:.2f}%")
    print(f"\nImage-level operating characteristic across the TAU grid "
          f"({len(unique_images)} Validation images):\n")
    print(detector_df.round(4).to_string(index=False))

    detector_df["youden_j"] = sum(
        detector_df[f"sensitivity_{n}"] - detector_df[f"false_alarm_{n}"] for n in GATED_CLASSES
    ) / len(GATED_CLASSES)
    sensitivity_ok = np.all(
        [detector_df[f"sensitivity_{n}"].to_numpy() >= SENSITIVITY_FLOOR for n in GATED_CLASSES], axis=0)
    cost_ok = detector_df["val_cost_points"].to_numpy() <= MAX_VALIDATION_COST
    eligible = detector_df[sensitivity_ok & cost_ok]
    if eligible.empty:
        raise RuntimeError(
            f"No TAU keeps sensitivity >= {SENSITIVITY_FLOOR} within a "
            f"{MAX_VALIDATION_COST}-point Validation budget. Read the table above before "
            "relaxing anything: if the detector cannot separate lesion from no-lesion at "
            "the image level, this gate is the wrong instrument and the backbone change "
            "(DINOv2-Small, already at 87.17% holdout in this project) is the better bet.")
    best = eligible.sort_values(["youden_j", "tau"], ascending=[False, False]).iloc[0]
    TAU = float(best["tau"])

    print(f"\n  Admissible TAU values: {eligible['tau'].tolist()}")
    print(f"  Selected TAU = {TAU:.2f} by max Youden J = {best['youden_j']:.4f}")
    for n in GATED_CLASSES:
        print(f"    {n}: sensitivity {best['sensitivity_' + n]:.3f}, "
              f"false alarm {best['false_alarm_' + n]:.3f}, "
              f"correct-veto rate {best['correct_veto_rate_' + n]:.3f}")
    print(f"  In-distribution cost at this TAU: {best['val_cost_points']:.2f} points "
          f"(budget {MAX_VALIDATION_COST})")

    save_json(OUT_DIR / "tau_selection.json", {
        "tau": TAU, "gated_classes": GATED_CLASSES,
        "rule": ("maximum Youden J (sensitivity - false alarm) over the TAU grid, among "
                 f"values keeping image-level sensitivity >= {SENSITIVITY_FLOOR} and "
                 f"Validation cost <= {MAX_VALIDATION_COST} points"),
        "youden_j": float(best["youden_j"]),
        "sensitivity_floor": SENSITIVITY_FLOOR,
        "max_validation_cost_points": MAX_VALIDATION_COST,
        "detector_sensitivity": {n: float(best[f"sensitivity_{n}"]) for n in GATED_CLASSES},
        "detector_false_alarm": {n: float(best[f"false_alarm_{n}"]) for n in GATED_CLASSES},
        "validation_baseline": baseline_val,
        "validation_gated": float(best["val_accuracy_mean"]),
        "validation_cost_points": float(best["val_cost_points"]),
        "holdout_was_used_for_selection": False,
    })
else:
    TAU = float(json.loads((OUT_DIR / "tau_selection.json").read_text(encoding="utf-8"))["tau"])
    print("Loaded TAU =", TAU)


Image-level operating characteristic on 1361 unique Validation images
  true anthracnose: 287 images | other: 1074
Classifier Validation baseline (ungated): 97.49%

Image-level operating characteristic across the TAU grid (1361 Validation images):

 tau  sensitivity_anthracnose  false_alarm_anthracnose  correct_veto_rate_anthracnose  val_accuracy_mean  val_cost_points
0.05                   1.0000                   0.6862                         0.3138             0.9760          -0.1176
0.10                   0.9895                   0.6294                         0.3706             0.9775          -0.2645
0.15                   0.9895                   0.5754                         0.4246             0.9781          -0.3233
0.20                   0.9861                   0.5428                         0.4572             0.9781          -0.3233
0.25                   0.9861                   0.5056                         0.4944             0.9781          -0.3233
0.30               

## Final evaluation

The gate is applied to **both** sides of the gap.

In [10]:
# ============================================================
# 06. FINAL EVALUATION — gate applied to BOTH Test and Holdout 09
# ============================================================
# The headline number of this project is a gap, so the gate must be applied to
# both sides of it. A gate that lifts Test as much as Holdout closes nothing,
# and reporting only the holdout side would overstate the result.

if RUN_FINAL_EVALUATION:
    if not (RUN_LEAKAGE_AUDIT and RUN_CALIBRATION):
        raise RuntimeError("Final evaluation requires a completed leakage audit and TAU calibration")
    if json.loads((OUT_DIR / "leakage_audit.json").read_text(encoding="utf-8"))["status"] != "PASS":
        raise RuntimeError("Leakage audit did not PASS")

    results = []
    for split in ("test", "holdout"):
        for seed in SEEDS:
            subset = predictions_df[(predictions_df["split"] == split) &
                                    (predictions_df["seed"] == seed)]
            truth = subset["true_label"].to_numpy()
            base = subset["baseline_pred"].to_numpy()
            gated, changed = apply_gate(subset, TAU)
            results.append({
                "split": split, "seed": seed, "images": len(subset),
                "baseline_accuracy": float((base == truth).mean()),
                "gated_accuracy": float((gated == truth).mean()),
                "baseline_macro_f1": float(f1_score(truth, base, average="macro", zero_division=0)),
                "gated_macro_f1": float(f1_score(truth, gated, average="macro", zero_division=0)),
                "vetoes": int(changed.sum()),
                "veto_helped": int(((gated == truth) & (base != truth) & changed).sum()),
                "veto_hurt": int(((gated != truth) & (base == truth) & changed).sum()),
            })
    results_df = pd.DataFrame(results)
    results_df.to_csv(OUT_DIR / "gated_results_5seeds.csv", index=False)

    def mean_of(split, column):
        return results_df[results_df["split"] == split][column].mean() * 100

    base_gap = mean_of("test", "baseline_accuracy") - mean_of("holdout", "baseline_accuracy")
    gated_gap = mean_of("test", "gated_accuracy") - mean_of("holdout", "gated_accuracy")

    print(f"TAU = {TAU:.2f}, gated classes: {GATED_CLASSES}\n")
    print(f"{'':<10}{'baseline':>22}{'gated':>22}{'delta':>10}")
    for split in ("test", "holdout"):
        subset = results_df[results_df["split"] == split]
        b, g = subset["baseline_accuracy"], subset["gated_accuracy"]
        print(f"  {split:<8}{100*b.mean():>10.2f} +/- {100*b.std(ddof=1):<6.2f}"
              f"{100*g.mean():>12.2f} +/- {100*g.std(ddof=1):<6.2f}"
              f"{100*(g.mean()-b.mean()):>+10.2f}")
    print(f"\n  GAP Test - Holdout: {base_gap:.2f}  ->  {gated_gap:.2f}  ({gated_gap-base_gap:+.2f})")
    print(f"  Target < 10.00 : {'MET' if gated_gap < 10 else 'NOT MET, short by ' + format(gated_gap - 10, '.2f')}")

    holdout = results_df[results_df["split"] == "holdout"]
    delta = (holdout["gated_accuracy"] - holdout["baseline_accuracy"]).to_numpy() * 100
    statistic, pvalue = stats.ttest_rel(holdout["gated_accuracy"], holdout["baseline_accuracy"])
    print(f"\n  Holdout, paired by seed: {delta.mean():+.2f} points, "
          f"{int((delta > 0).sum())}/5 seeds improved, t(4)={statistic:.3f}, p={pvalue:.4f}")
    print(f"  Per seed: {np.round(delta, 2).tolist()}")
    print(f"  Vetoes fired: Test {int(results_df[results_df['split']=='test']['vetoes'].sum())}, "
          f"Holdout {int(holdout['vetoes'].sum())} "
          f"(helped {int(holdout['veto_helped'].sum())} / hurt {int(holdout['veto_hurt'].sum())})")

    # Per-class effect on Holdout, pooled over seeds.
    pooled = predictions_df[predictions_df["split"] == "holdout"].copy()
    pooled["gated_pred"], _ = apply_gate(pooled, TAU)
    per_class = []
    for name in CLASS_NAMES:
        mask = pooled["true_label"] == name
        if not mask.any():
            continue
        per_class.append({
            "class": name, "support": int(mask.sum()),
            "baseline_recall": float((pooled.loc[mask, "baseline_pred"] == name).mean()),
            "gated_recall": float((pooled.loc[mask, "gated_pred"] == name).mean()),
            "baseline_predicted": int((pooled["baseline_pred"] == name).sum()),
            "gated_predicted": int((pooled["gated_pred"] == name).sum()),
        })
    per_class_df = pd.DataFrame(per_class)
    per_class_df["recall_delta"] = (per_class_df["gated_recall"] - per_class_df["baseline_recall"]) * 100
    per_class_df.to_csv(OUT_DIR / "gated_per_class_holdout.csv", index=False)
    print("\nHoldout per-class (pooled over 5 seeds):")
    print(per_class_df.to_string(index=False))

    save_json(OUT_DIR / "exp031_summary.json", {
        "experiment_id": EXPERIMENT_ID, "baseline_experiment": BASELINE_EXPERIMENT_ID,
        "detector": DETECTOR_NAME, "tau": TAU, "gated_classes": GATED_CLASSES,
        "test_baseline": mean_of("test", "baseline_accuracy"),
        "test_gated": mean_of("test", "gated_accuracy"),
        "holdout_baseline": mean_of("holdout", "baseline_accuracy"),
        "holdout_gated": mean_of("holdout", "gated_accuracy"),
        "gap_baseline": base_gap, "gap_gated": gated_gap,
        "gap_target_met": bool(gated_gap < 10),
        "holdout_paired_t": float(statistic), "holdout_paired_p": float(pvalue),
        "seeds_improved": int((delta > 0).sum()),
        "caveats": [
            "Holdout 09 has 126 images and 4 ground-truth classes; 95% CI at 86% is about +/-6 points.",
            "Test and Holdout 09 are historically exposed; this is a descriptive audit, not a blind benchmark.",
            "TAU was selected on Validation only, with detector-seen images removed.",
            "real_healthy_40 and _42 share SHA256 with real_anthracnose_157 and _159 from Train V12; "
            "their labels need expert review before any Healthy conclusion.",
        ],
    })
    print("\nWrote exp031_summary.json")
else:
    print("Final evaluation NOT_EXECUTED")


TAU = 0.60, gated classes: ['anthracnose']

                        baseline                 gated     delta
  test         96.23 +/- 0.49         94.54 +/- 0.30       -1.69
  holdout      82.70 +/- 1.30         86.35 +/- 1.89       +3.65

  GAP Test - Holdout: 13.54  ->  8.19  (-5.34)
  Target < 10.00 : MET

  Holdout, paired by seed: +3.65 points, 5/5 seeds improved, t(4)=6.782, p=0.0025
  Per seed: [4.76, 2.38, 3.97, 2.38, 4.76]
  Vetoes fired: Test 129, Holdout 74 (helped 43 / hurt 20)

Holdout per-class (pooled over 5 seeds):
      class  support  baseline_recall  gated_recall  baseline_predicted  gated_predicted  recall_delta
anthracnose      230         0.873913      0.786957                 278              204     -8.695652
    healthy       95         0.652632      0.936842                  72              117     28.421053
 leaf_miner      180         0.916667      0.927778                 170              172      1.111111
   red_rust      125         0.744000      0.856000

## Archive and upload EXP-031 results

After the final evaluation, this cell creates one `RESULTS.zip` and uploads it directly to the confirmed `current_dataset` Drive folder. The upload is checked by size, MD5 and parent folder.


In [11]:
"""Archive EXP-031 results and upload the ZIP to the confirmed Drive folder."""

if RUN_FINAL_EVALUATION:
    if not (OUT_DIR / "exp031_summary.json").is_file():
        raise RuntimeError("EXP-031 final evaluation is incomplete; refusing to export results")
    result_files = sorted(path for path in OUT_DIR.rglob("*") if path.is_file())
    if not result_files:
        raise RuntimeError("EXP-031 has no result files")
    manifest = [{"path": path.relative_to(OUT_DIR).as_posix(),
                 "bytes": path.stat().st_size, "sha256": file_sha256(path)}
                for path in result_files]
    save_json(OUT_DIR / "artifact_manifest.json", manifest)
    result_zip = OUT_DIR.parent / f"{EXPERIMENT_ID}_RESULTS.zip"
    with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=6, allowZip64=True) as archive:
        for path in sorted(OUT_DIR.rglob("*")):
            if path.is_file():
                archive.write(path, f"{EXPERIMENT_ID}/{path.relative_to(OUT_DIR).as_posix()}")
    with zipfile.ZipFile(result_zip) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("EXP-031 result ZIP failed integrity check")
    print("Created:", result_zip, "bytes:", result_zip.stat().st_size,
          "SHA256:", file_sha256(result_zip))

    if UPLOAD_RESULT_ZIPS_TO_DRIVE:
        import google.auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaFileUpload
        if Path("/content").is_dir():
            from google.colab import auth
            auth.authenticate_user()
        credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
        service = build("drive", "v3", credentials=credentials, cache_discovery=False)
        folder = service.files().get(fileId=DRIVE_TARGET_FOLDER_ID,
                                     fields="id,name,mimeType,trashed",
                                     supportsAllDrives=True).execute()
        if (folder.get("mimeType") != "application/vnd.google-apps.folder" or
                folder.get("trashed")):
            raise RuntimeError(f"Invalid Drive ZIP destination: {DRIVE_TARGET_FOLDER_ID}")
        query = (f"name = '{result_zip.name}' and "
                 f"'{DRIVE_TARGET_FOLDER_ID}' in parents and trashed = false")
        found = service.files().list(q=query, fields="nextPageToken,files(id,name,size,md5Checksum)",
                                     pageSize=100, supportsAllDrives=True,
                                     includeItemsFromAllDrives=True).execute()
        matches = found.get("files", [])
        if found.get("nextPageToken") or len(matches) > 1:
            raise RuntimeError(f"Ambiguous Drive archive: {result_zip.name}")
        digest = hashlib.md5()
        with result_zip.open("rb") as stream:
            for chunk in iter(lambda: stream.read(8 << 20), b""):
                digest.update(chunk)
        local_md5 = digest.hexdigest()
        if matches and int(matches[0].get("size", -1)) == result_zip.stat().st_size and \
                matches[0].get("md5Checksum") == local_md5:
            file_id = matches[0]["id"]
        else:
            media = MediaFileUpload(str(result_zip), mimetype="application/zip",
                                    resumable=True, chunksize=8 << 20)
            request = (service.files().update(fileId=matches[0]["id"], media_body=media,
                                              fields="id", supportsAllDrives=True) if matches else
                       service.files().create(body={"name": result_zip.name,
                                                    "parents": [DRIVE_TARGET_FOLDER_ID]},
                                              media_body=media, fields="id",
                                              supportsAllDrives=True))
            uploaded = None
            while uploaded is None:
                progress, uploaded = request.next_chunk(num_retries=3)
                if progress:
                    print(result_zip.name, f"{progress.progress():.0%}")
            file_id = uploaded["id"]
        remote = service.files().get(fileId=file_id,
            fields="id,name,size,md5Checksum,parents,webViewLink",
            supportsAllDrives=True).execute()
        if (remote["name"] != result_zip.name or
                int(remote["size"]) != result_zip.stat().st_size or
                remote.get("md5Checksum") != local_md5 or
                DRIVE_TARGET_FOLDER_ID not in remote.get("parents", [])):
            raise RuntimeError(f"Drive upload verification failed: {result_zip.name}")
        print("Verified Drive archive:", remote.get("webViewLink") or
              f"https://drive.google.com/file/d/{file_id}/view")
else:
    print("EXP-031 result ZIP pending until final evaluation")


Created: /content/training_results/current_dataset/EXP-DENSENET121-YOLOGATE-031_RESULTS.zip bytes: 178043 SHA256: 1d80721262ebe3eeb34b33e97236e5accad7b8ee3cc8f3cbf50887b92cee9211


Verified Drive archive: https://drive.google.com/file/d/14freRE5QL2nuA7fXvawdroYMGXlQ32ya/view?usp=drivesdk
